In [0]:
dbutils.widgets.text("batch_id","")
v_batch_id = dbutils.widgets.get("batch_id")

In [0]:
%run ../0-common/env-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"

silver_table = f"{catalog_name}.{silver_schema}.sprints"

In [0]:
from pyspark.sql import functions as F

In [0]:
sprints_df = (
    spark.read.table(bronze_table).filter(F.col("batch_id") == v_batch_id)
    )

In [0]:
sprints_selected_df = sprints_df.drop("url")

In [0]:
sprints_renamed_df = (
    sprints_selected_df
    .withColumnsRenamed({
        "raceName": "race_name",
        "constructorId": "constructor_id",
        "driverId": "driver_id",
        "positionText": "position_text",
        "date": "race_date",
        "grid": "grid_position",
        "number": "car_number",
        "position": "final_position",
        "positionText": "final_position_text"
    })  
)

In [0]:
sprints_distinct_df = (
    sprints_renamed_df
    .filter(
        F.col("round").isNotNull() &
        F.col("season").isNotNull() &
        F.col("constructor_id").isNotNull() &
        F.col("driver_id").isNotNull()
    )
    .dropDuplicates(["round", "season", "constructor_id", "driver_id"])
)

In [0]:
sprints_final_df = (
    sprints_distinct_df
    .withColumn("race_name", F.initcap("race_name"))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("updated_at", F.current_timestamp())
)

In [0]:
if not spark.catalog.tableExists(silver_table):
    (
        sprints_final_df.write
        .format('delta')
        .mode("overwrite")
        .saveAsTable(silver_table)
    )
else:
    from delta.tables import DeltaTable

    delta_table = DeltaTable.forName(spark, silver_table)
    (
        delta_table.alias("t")
        .merge(
            sprints_final_df.alias("s"),
            "t.season = s.season AND t.round = s.round AND t.constructor_id = s.constructor_id AND t.driver_id = s.driver_id"
        )
        .whenMatchedUpdate(
            condition="s.batch_id >= t.batch_id",
            set={
                "race_date": "s.race_date",
                "race_name": "s.race_name",
                "grid_position": "s.grid_position",
                "laps": "s.laps",
                "car_number": "s.car_number",
                "points": "s.points",
                "final_position": "s.final_position",
                "final_position_text": "s.final_position_text",
                "status": "s.status",
                "ingestion_timestamp": "s.ingestion_timestamp",
                "source_file": "s.source_file",
                "batch_id": "s.batch_id",
                "updated_at": "s.updated_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )